In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    train_test_split,
    cross_val_predict,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
MIN_RECALL_CLASS1 = 0.50
np.random.seed(SEED)

#Guarding against trivial results by forcing recall > 0.5
def precision_with_min_recall(estimator, X, y_true):
    y_pred = estimator.predict(X)
    recall = recall_score(y_true, y_pred, zero_division=0)
    if recall < MIN_RECALL_CLASS1:
        return 0.0
    return precision_score(y_true, y_pred, zero_division=0)

In [ ]:
DATA_PATH = "merged_df_cleaned.csv"

df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")
df.columns = [c.strip() for c in df.columns]

df = df.dropna(subset=["survived"]).copy()
df["survived"] = df["survived"].astype(int)

print("Dataset shape:", df.shape)
print("Target distribution:")
print(df["survived"].value_counts(normalize=True).rename("ratio"))

Dataset shape: (125922, 38)
Target distribution:
survived
1    0.794555
0    0.205445
Name: ratio, dtype: float64


In [ ]:
identifier_cols = [c for c in ["store_id", "name", "address", "genres_list"] if c in df.columns]

X = df.drop(columns=["survived"] + identifier_cols)
y = df["survived"]

categorical_cols = X.select_dtypes(include=["object", "category", "string"]).columns.tolist()
numeric_cols = X.select_dtypes(exclude=["object", "category", "string"]).columns.tolist()

print(f"Numeric features: {len(numeric_cols)}")
print(f"Categorical features: {len(categorical_cols)}")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEED,
    stratify=y,
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Numeric features: 26
Categorical features: 7
Train shape: (100737, 33)
Test shape: (25185, 33)


In [ ]:
#Using same preprocessing for all columns. Previously tested to generally be best 
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=30)),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_cols),
    ("cat", categorical_pipe, categorical_cols),
], remainder="drop")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

In [ ]:
searches = {
    "logistic_regression": GridSearchCV(
        estimator=Pipeline([
            ("prep", preprocessor),
            ("model", LogisticRegression(
                max_iter=1500,
                class_weight="balanced",
                solver="liblinear",
                random_state=SEED,
            )),
        ]),
        param_grid={"model__C": [0.2, 1.0, 5.0]},
        scoring=precision_with_min_recall,
        cv=cv,
        n_jobs=1,
        verbose=1,
    ),
    "random_forest": GridSearchCV(
        estimator=Pipeline([
            ("prep", preprocessor),
            ("model", RandomForestClassifier(
                random_state=SEED,
                n_jobs=1,
                class_weight="balanced_subsample",
            )),
        ]),
        param_grid={
            "model__n_estimators": [300],
            "model__max_depth": [None, 25],
            "model__min_samples_leaf": [1, 3],
        },
        scoring=precision_with_min_recall,
        cv=cv,
        n_jobs=1,
        verbose=1,
    ),
    "extra_trees": GridSearchCV(
        estimator=Pipeline([
            ("prep", preprocessor),
            ("model", ExtraTreesClassifier(
                random_state=SEED,
                n_jobs=1,
                class_weight="balanced",
            )),
        ]),
        param_grid={
            "model__n_estimators": [400],
            "model__max_depth": [None, 30],
            "model__min_samples_leaf": [1, 3],
        },
        scoring=precision_with_min_recall,
        cv=cv,
        n_jobs=1,
        verbose=1,
    ),
}

results = []
best_name = None
best_search = None
best_cv_precision = -np.inf

for name, search in searches.items():
    print(f"=== Training: {name} ===")
    search.fit(X_train, y_train)
    score = float(search.best_score_)

    results.append({
        "model": name,
        "best_cv_constrained_precision_class1": score,
        "best_params": search.best_params_,
    })

    print(f"Best CV precision class 1 ({name}): {score:.4f}")
    print(f"Best params ({name}): {search.best_params_}")

    if score > best_cv_precision:
        best_cv_precision = score
        best_name = name
        best_search = search

results_df = pd.DataFrame(results).sort_values("best_cv_constrained_precision_class1", ascending=False)
print("\nModel ranking by constrained CV precision (class 1, recall >= 0.5):")
print(results_df[["model", "best_cv_constrained_precision_class1"]])
print(f"\nSelected model: {best_name} (constrained CV precision class 1 = {best_cv_precision:.4f})")

=== Training: logistic_regression ===
Fitting 5 folds for each of 3 candidates, totalling 15 fits
Best CV precision class 1 (logistic_regression): 0.8439
Best params (logistic_regression): {'model__C': 0.2}
=== Training: random_forest ===
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Best CV precision class 1 (random_forest): 0.8497
Best params (random_forest): {'model__max_depth': 25, 'model__min_samples_leaf': 3, 'model__n_estimators': 300}
=== Training: extra_trees ===
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Best CV precision class 1 (extra_trees): 0.8412
Best params (extra_trees): {'model__max_depth': 30, 'model__min_samples_leaf': 3, 'model__n_estimators': 400}

Model ranking by constrained CV precision (class 1, recall >= 0.5):
                 model  best_cv_constrained_precision_class1
1        random_forest                              0.849718
0  logistic_regression                              0.843865
2          extra_trees                 

In [ ]:
best_model = best_search.best_estimator_

oof_proba = cross_val_predict(
    best_model,
    X_train,
    y_train,
    cv=cv,
    method="predict_proba",
    n_jobs=1,
)[:, 1]

thresholds = np.linspace(0.05, 0.95, 181)
threshold_metrics = []

for t in thresholds:
    pred = (oof_proba >= t).astype(int)
    precision_val = precision_score(y_train, pred, zero_division=0)
    recall_val = recall_score(y_train, pred, zero_division=0)
    threshold_metrics.append({
        "threshold": float(t),
        "precision": float(precision_val),
        "recall": float(recall_val),
    })

feasible_metrics = [m for m in threshold_metrics if m["recall"] >= MIN_RECALL_CLASS1]

if not feasible_metrics:
    raise ValueError("No threshold satisfied the minimum class-1 recall requirement.")

best_metric = max(feasible_metrics, key=lambda m: (m["precision"], m["recall"]))
best_threshold = best_metric["threshold"]

print("Best threshold (OOF, constrained precision):", round(best_threshold, 4))
print("Best OOF precision (class 1):", round(float(best_metric["precision"]), 4))
print("OOF recall at selected threshold (class 1):", round(float(best_metric["recall"]), 4))

Best threshold (OOF, constrained precision): 0.515
Best OOF precision (class 1): 0.8622
OOF recall at selected threshold (class 1): 0.5262


In [ ]:
test_proba = best_model.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= best_threshold).astype(int)

test_precision_class1 = precision_score(y_test, test_pred, zero_division=0)
test_recall_class1 = recall_score(y_test, test_pred, zero_division=0)
test_f1_class1 = f1_score(y_test, test_pred, zero_division=0)
test_accuracy = accuracy_score(y_test, test_pred)

if test_recall_class1 < MIN_RECALL_CLASS1:
    print("Warning: holdout recall for class 1 is below the requested minimum.")

print("Selected model:", best_name)
print("Estimated holdout test precision (class 1):", round(float(test_precision_class1), 4))
print("Estimated holdout test recall (class 1):", round(float(test_recall_class1), 4))
print("Estimated holdout test F1 (class 1):", round(float(test_f1_class1), 4))
print("Estimated holdout test accuracy:", round(float(test_accuracy), 4))

print("Confusion matrix:")
print(confusion_matrix(y_test, test_pred))

print("Classification report:")
print(classification_report(y_test, test_pred, digits=4))

Selected model: random_forest
Estimated holdout test precision (class 1): 0.8634
Estimated holdout test recall (class 1): 0.5111
Estimated holdout test F1 (class 1): 0.6421
Estimated holdout test accuracy: 0.5473
Confusion matrix:
[[ 3556  1618]
 [ 9783 10228]]
Classification report:
              precision    recall  f1-score   support

           0     0.2666    0.6873    0.3842      5174
           1     0.8634    0.5111    0.6421     20011

    accuracy                         0.5473     25185
   macro avg     0.5650    0.5992    0.5131     25185
weighted avg     0.7408    0.5473    0.5891     25185

